# Data Ingestion

This notebook collects and stores the raw market data used in the project.

The analysis models a crypto payment operation in which **incoming payments received in volatile cryptocurrencies are converted into USDT to reduce exposure to short-term price movements**.

The project then examines how this operational balance could behave during different historical market events and what risks may remain after the conversion.

Two analytical layers are used:

1. **BTC and ETH — broader market context**  
   These assets represent the volatile side of the model and are used to show how strongly the broader crypto market was moving around each event.

2. **USDT — modelled operational balance**  
   After conversion, incoming funds are modelled as being held in USDT. **USDC is included as a comparison stablecoin** to help distinguish broader stablecoin-market stress from movements specific to one asset.

The project uses several market data sources:

- **yfinance** — long-term daily BTC, ETH, USDT and USDC data;
- **Binance public API** — hourly BTC and ETH market data;
- **Coinbase Exchange API** — hourly USDT/EUR and USDC/EUR data;
- **Dukascopy Historical Data Export** — hourly EUR/USD FX benchmark data.

Raw data is stored without analytical transformation. Cleaning, validation and schema standardization are performed in the next notebook.

In [51]:
import pandas as pd
import yfinance as yf
import requests

## 1. Daily Market Data — yfinance

Daily BTC, ETH, USDT and USDC data is collected from 2020 onward to provide a broader historical view of how these assets behave.

The daily dataset is later used to place the selected event periods in a wider market context.

The original yfinance column structure is preserved when saving the raw dataset.

In [52]:
crypto_daily = yf.download(
    ["BTC-USD", "ETH-USD", 'USDT-USD', 'USDC-USD'],
    start="2020-01-01",
    end="2026-08-20",
    interval="1d",
    auto_adjust=False
)

crypto_daily.head()

[*********************100%***********************]  4 of 4 completed


Price         Adj Close                                        Close  \
Ticker          BTC-USD     ETH-USD  USDC-USD  USDT-USD      BTC-USD   
Date                                                                   
2020-01-01  7200.174316  130.802002  1.004079  0.999836  7200.174316   
2020-01-02  6985.470215  127.410179  1.005017  1.001565  6985.470215   
2020-01-03  7344.884277  134.171707  1.005273  1.004192  7344.884277   
2020-01-04  7410.656738  135.069366  1.009466  1.007472  7410.656738   
2020-01-05  7411.317383  136.276779  1.008497  1.006197  7411.317383   

Price                                              High              ...  \
Ticker         ETH-USD  USDC-USD  USDT-USD      BTC-USD     ETH-USD  ...   
Date                                                                 ...   
2020-01-01  130.802002  1.004079  0.999836  7254.330566  132.835358  ...   
2020-01-02  127.410179  1.005017  1.001565  7212.155273  130.820038  ...   
2020-01-03  134.171707  1.005273  1.004192  7413.715332  134.554016  ...   
2020-01-04  135.069366  1.009466  1.007472  7427.385742  136.052719  ...   
2020-01-05  136.276779  1.008497  1.006197  7544.497070  139.410202  ...   

Price            Low                   Open                                  \
Ticker      USDC-USD  USDT-USD      BTC-USD     ETH-USD  USDC-USD  USDT-USD   
Date                                                                          
2020-01-01  1.001989  0.994924  7194.892090  129.630661  1.003730  0.999571   
2020-01-02  1.002543  0.986515  7202.551270  130.820038  1.004431  0.999788   
2020-01-03  0.988455  0.988027  6984.428711  127.411263  1.005357  1.001183   
2020-01-04  1.001273  0.999160  7345.375488  134.168518  1.004818  1.003510   
2020-01-05  1.003932  1.001758  7410.451660  135.072098  1.008748  1.009921   

Price            Volume                                       
Ticker          BTC-USD      ETH-USD   USDC-USD     USDT-USD  
Date                                                          
2020-01-01  18565664997   7935230330  242586528  21503143454  
2020-01-02  20802083465   8032709256  318268134  24212314977  
2020-01-03  28111481032  10476845358  374792167  32420287856  
2020-01-04  18444271275   7430904515  334494308  21585629320  
2020-01-05  19725074095   7526675353  334597544  24090142146  

[5 rows x 24 columns]

In [53]:
crypto_daily.to_csv("../data/raw/daily/crypto_daily_full.csv")

## 2. Hourly Crypto-Market Data — Binance

Daily data provides the broader market picture, but major crypto-market movements can develop within hours.

For this reason, hourly BTC and ETH data is collected from the Binance public API around each selected historical event.

BTC and ETH are not treated as the operational balance in this project. Their role is to show the **timing and magnitude of broader crypto-market movements** around each event.

The raw API response is preserved with all available Binance fields. Timestamp conversion, field selection, numeric conversion and validation are performed later in the cleaning stage.

### Event Window Definition

A standardized extraction window is applied to all four historical events to provide sufficient data before and after each event.

Hourly BTC and ETH data is collected for approximately one month around each event.

The extraction window is intentionally wider than the final analytical window. During event-specific exploration, the relevant period may be narrowed once the event timeline and T0 are defined.

This approach allows the raw data layer to remain consistent while each event analysis focuses only on the period required to answer its specific question.

Long-term market behaviour is assessed separately using the daily historical dataset.

In [54]:
binance_url = "https://api.binance.com/api/v3/klines"

In [55]:
def get_binance_hourly_raw(symbol, start_date, end_date):
    start_ts = int(
        pd.Timestamp(start_date, tz="UTC").timestamp() * 1000
    )
    end_ts = int(
        pd.Timestamp(end_date, tz="UTC").timestamp() * 1000
    )

    params = {
        "symbol": symbol,
        "interval": "1h",
        "startTime": start_ts,
        "endTime": end_ts,
        "limit": 1000
    }

    response = requests.get(
        binance_url,
        params=params,
        timeout=30
    )
    response.raise_for_status()

    data = response.json()

    columns = [
        "open_time",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "close_time",
        "quote_volume",
        "trades",
        "taker_buy_base",
        "taker_buy_quote",
        "ignore"
    ]

    return pd.DataFrame(data, columns=columns)

In [56]:
btc_luna_raw = get_binance_hourly_raw(
    "BTCUSDT",
    "2022-04-25", 
    "2022-05-25"
)
btc_luna_raw.head()

,open_time,open,high,low,close,volume,close_time,quote_volume,trades,taker_buy_base,taker_buy_quote,ignore
0,1650844800000,39450.12000000,39500.84000000,38674.00000000,38855.47000000,4228.78013000,1650848399999,165079562.17369930,81697,1807.04285000,70534732.76962680,0
1,1650848400000,38855.47000000,39154.38000000,38747.51000000,39095.81000000,2336.46392000,1650851999999,91004528.73470060,46594,1087.17006000,42341565.98327780,0
2,1650852000000,39095.81000000,39153.94000000,38961.64000000,39091.17000000,1205.51583000,1650855599999,47084219.77242880,27848,597.02926000,23317634.38112080,0
3,1650855600000,39091.17000000,39294.76000000,39086.37000000,39253.71000000,1443.33178000,1650859199999,56585011.34388490,29441,757.04103000,29677144.12016310,0
4,1650859200000,39253.70000000,39256.28000000,39055.71000000,39139.74000000,896.85541000,1650862799999,35117921.48971000,21447,405.26210000,15868560.25020230,0


In [57]:
btc_luna_raw.to_csv(
    "../data/raw/hourly/binance/btc_luna_hourly.csv",
    index=False
)

In [58]:
eth_luna_raw = get_binance_hourly_raw(
    "ETHUSDT",
    "2022-04-25", 
    "2022-05-25"
)
eth_luna_raw.head()

,open_time,open,high,low,close,volume,close_time,quote_volume,trades,taker_buy_base,taker_buy_quote,ignore
0,1650844800000,2920.99000000,2925.75000000,2851.40000000,2856.55000000,54637.71530000,1650848399999,157562279.33629800,70377,21530.87580000,62060549.97802500,0
1,1650848400000,2856.55000000,2868.97000000,2841.60000000,2865.04000000,24748.35260000,1650851999999,70676630.89477900,36933,11438.30390000,32673441.65384100,0
2,1650852000000,2865.03000000,2869.45000000,2853.18000000,2860.44000000,14716.03500000,1650855599999,42106132.47044900,19592,7778.55900000,22256426.26012800,0
3,1650855600000,2860.43000000,2879.72000000,2860.00000000,2875.59000000,11779.16770000,1650859199999,33840831.57491300,18166,6740.26180000,19364588.84723600,0
4,1650859200000,2875.58000000,2876.00000000,2863.38000000,2864.99000000,13012.45320000,1650862799999,37314659.88727200,15686,6533.16330000,18729262.98868100,0


In [59]:
eth_luna_raw.to_csv(
    "../data/raw/hourly/binance/eth_luna_hourly.csv",
    index=False
)

In [60]:
btc_celsius_raw = get_binance_hourly_raw(
    "BTCUSDT",
    "2022-05-29", 
    "2022-06-28"
)
btc_celsius_raw.head()

,open_time,open,high,low,close,volume,close_time,quote_volume,trades,taker_buy_base,taker_buy_quote,ignore
0,1653782400000,29031.33000000,29045.34000000,28970.30000000,29005.47000000,786.45225000,1653785999999,22815015.13628360,22692,279.37712000,8105065.51108810,0
1,1653786000000,29005.46000000,29024.65000000,28900.00000000,28925.85000000,940.66592000,1653789599999,27241164.12748350,24640,368.15723000,10661098.23851050,0
2,1653789600000,28925.85000000,28925.86000000,28839.21000000,28904.13000000,1227.04778000,1653793199999,35444026.01929270,27383,596.15271000,17219356.14659530,0
3,1653793200000,28904.12000000,28997.21000000,28876.08000000,28984.32000000,719.15298000,1653796799999,20806485.73978020,18690,351.82046000,10178687.45429560,0
4,1653796800000,28984.31000000,28984.32000000,28914.96000000,28941.09000000,456.89978000,1653800399999,13229951.38933500,16648,174.68137000,5058017.15060690,0


In [61]:
btc_celsius_raw.to_csv(
    "../data/raw/hourly/binance/btc_celsius_hourly.csv",
    index=False
)

In [62]:
eth_celsius_raw = get_binance_hourly_raw(
    "ETHUSDT",
    "2022-05-29", 
    "2022-06-28"
)
eth_celsius_raw.head()

,open_time,open,high,low,close,volume,close_time,quote_volume,trades,taker_buy_base,taker_buy_quote,ignore
0,1653782400000,1792.21000000,1795.89000000,1786.78000000,1789.66000000,16687.71100000,1653785999999,29897604.40560700,17785,7434.14880000,13317646.74946000,0
1,1653786000000,1789.67000000,1789.80000000,1772.82000000,1776.23000000,23647.59810000,1653789599999,42103062.22166500,25599,9874.01830000,17582418.16876400,0
2,1653789600000,1776.22000000,1776.74000000,1759.96000000,1768.57000000,36374.21470000,1653793199999,64241691.16960700,26047,21190.44680000,37400412.69412200,0
3,1653793200000,1768.56000000,1778.52000000,1765.01000000,1776.46000000,12530.06340000,1653796799999,22202010.42332100,14698,6242.28870000,11059732.29628600,0
4,1653796800000,1776.46000000,1779.28000000,1769.38000000,1775.26000000,10996.77650000,1653800399999,19516636.36973400,12231,5295.06830000,9396745.83373300,0


In [63]:
eth_celsius_raw.to_csv(
    "../data/raw/hourly/binance/eth_celsius_hourly.csv",
    index=False
)

In [64]:
btc_svb_raw = get_binance_hourly_raw(
    "BTCUSDT",
    "2023-02-24",
    "2023-03-26" 
)
btc_svb_raw.head()

,open_time,open,high,low,close,volume,close_time,quote_volume,trades,taker_buy_base,taker_buy_quote,ignore
0,1677196800000,23940.20000000,24022.87000000,23890.00000000,23958.71000000,9447.10151000,1677200399999,226243534.24398220,330472,4658.01245000,111561025.31241050,0
1,1677200400000,23958.71000000,24132.35000000,23956.80000000,24002.85000000,13218.58218000,1677203999999,317931133.60612160,422749,6730.47207000,161882203.18754170,0
2,1677204000000,24001.78000000,24029.91000000,23907.15000000,23940.42000000,9441.36012000,1677207599999,226312342.47064430,331975,4557.69982000,109251804.69237280,0
3,1677207600000,23940.42000000,23998.10000000,23921.46000000,23954.05000000,5655.48755000,1677211199999,135513012.46656260,240557,2919.87143000,69965115.68037130,0
4,1677211200000,23954.50000000,23986.12000000,23923.88000000,23942.21000000,6471.52290000,1677214799999,155087728.73691360,243571,3181.68103000,76251933.66829210,0


In [65]:
btc_svb_raw.to_csv(
    "../data/raw/hourly/binance/btc_svb_hourly.csv",
    index=False
)

In [66]:
eth_svb_raw = get_binance_hourly_raw(
    "ETHUSDT",
    "2023-02-24",
    "2023-03-26" 
)
eth_svb_raw.head()

,open_time,open,high,low,close,volume,close_time,quote_volume,trades,taker_buy_base,taker_buy_quote,ignore
0,1677196800000,1650.52000000,1657.39000000,1646.22000000,1651.50000000,13641.47360000,1677200399999,22524333.56608500,29915,6961.84850000,11495462.85949800,0
1,1677200400000,1651.49000000,1664.66000000,1651.49000000,1653.24000000,18798.94000000,1677203999999,31173215.58958700,36308,9489.28180000,15736079.60519400,0
2,1677204000000,1653.25000000,1655.50000000,1646.34000000,1648.48000000,13473.94560000,1677207599999,22249598.42506700,24429,5744.45650000,9486893.92997800,0
3,1677207600000,1648.49000000,1651.88000000,1646.04000000,1649.14000000,8491.86470000,1677211199999,14007073.39108400,17328,4009.42830000,6613253.05764900,0
4,1677211200000,1649.15000000,1651.47000000,1646.82000000,1647.04000000,7366.98080000,1677214799999,12151162.08206400,14823,3473.77400000,5729483.46537400,0


In [67]:
eth_svb_raw.to_csv(
    "../data/raw/hourly/binance/eth_svb_hourly.csv",
    index=False
)

In [68]:
btc_election_raw = get_binance_hourly_raw(
    "BTCUSDT",
    "2024-10-23", 
    "2024-11-22"
)
btc_election_raw.head()

,open_time,open,high,low,close,volume,close_time,quote_volume,trades,taker_buy_base,taker_buy_quote,ignore
0,1729641600000,67426.01000000,67472.83000000,67130.43000000,67179.13000000,577.86445000,1729645199999,38856655.94929450,129987,259.03621000,17418389.77957100,0
1,1729645200000,67179.13000000,67386.94000000,67179.13000000,67348.46000000,393.55985000,1729648799999,26483283.42330240,71709,195.69149000,13166942.21772700,0
2,1729648800000,67348.46000000,67386.93000000,67068.28000000,67096.00000000,581.36673000,1729652399999,39062967.26570060,94457,200.62056000,13482233.95647130,0
3,1729652400000,67095.99000000,67180.92000000,66890.00000000,67082.10000000,639.25878000,1729655999999,42838409.11328390,111648,280.36798000,18787741.28228520,0
4,1729656000000,67082.10000000,67249.84000000,66999.99000000,67217.48000000,515.38779000,1729659599999,34598962.38305570,65328,286.67975000,19249765.03158070,0


In [69]:
btc_election_raw.to_csv(
    "../data/raw/hourly/binance/btc_election_hourly.csv",
    index=False
)

In [70]:
eth_election_raw = get_binance_hourly_raw(
    "ETHUSDT",
    "2024-10-23", 
    "2024-11-22"
)
eth_election_raw.head()

,open_time,open,high,low,close,volume,close_time,quote_volume,trades,taker_buy_base,taker_buy_quote,ignore
0,1729641600000,2622.81000000,2628.20000000,2611.92000000,2614.09000000,7051.04190000,1729645199999,18466532.54041500,68992,3124.35640000,8182848.12620100,0
1,1729645200000,2614.09000000,2625.43000000,2613.81000000,2624.16000000,4605.22810000,1729648799999,12071419.00809400,46811,2595.18660000,6801988.48034400,0
2,1729648800000,2624.14000000,2624.50000000,2610.37000000,2613.19000000,5899.93240000,1729652399999,15436638.97088500,54993,2234.65910000,5846240.01256600,0
3,1729652400000,2613.18000000,2617.75000000,2601.18000000,2616.50000000,10813.83120000,1729655999999,28225274.67910700,82248,5814.34290000,15174945.69651100,0
4,1729656000000,2616.49000000,2619.91000000,2610.07000000,2617.25000000,6008.54390000,1729659599999,15714967.01031700,53276,2964.56540000,7754194.85992800,0


In [71]:
eth_election_raw.to_csv(
    "../data/raw/hourly/binance/eth_election_hourly.csv",
    index=False
)

## 3. Hourly Stablecoin Data — Coinbase

Hourly USDT/EUR and USDC/EUR candlestick data is collected from the Coinbase Exchange public API for the same broader event periods.

In the modelled scenario, **USDT represents the operational balance after conversion from volatile cryptocurrencies**.

USDC is included as a **comparison stablecoin**. Comparing the two helps distinguish movements affecting the broader stablecoin market from events that may be specific to one stablecoin.

Coinbase provides these assets as EUR market pairs. Because USDT and USDC are designed to remain close to **$1**, their EUR prices are later combined with EUR/USD data to estimate their approximate USD value.

Coinbase limits a single candle request to a maximum of 300 observations. Longer event windows are therefore retrieved in smaller intervals and combined into one raw dataset per asset and event.

The raw API response is preserved at this stage. Timestamp conversion, numeric conversion, validation and analytical transformations are performed later.

In [72]:
coinbase_url = "https://api.exchange.coinbase.com"
coinbase_granularity = 3600  # 1 hour

In [73]:
def get_coinbase_hourly_raw(product_id, start_date, end_date):
    start = pd.Timestamp(start_date, tz="UTC")
    end = pd.Timestamp(end_date, tz="UTC")

    chunk_hours = 250
    chunks = []

    current_start = start

    while current_start < end:
        current_end = min(
            current_start + pd.Timedelta(hours=chunk_hours),
            end
        )

        url = f"{coinbase_url}/products/{product_id}/candles"

        params = {
            "start": current_start.isoformat(),
            "end": current_end.isoformat(),
            "granularity": coinbase_granularity
        }

        response = requests.get(
            url,
            params=params,
            timeout=30
        )

        response.raise_for_status()

        data = response.json()

        columns = [
            "timestamp",
            "low",
            "high",
            "open",
            "close",
            "volume"
        ]

        chunk_df = pd.DataFrame(
            data,
            columns=columns
        )
        chunks.append(chunk_df)

        current_start = current_end
    full_df = (pd.concat(chunks, ignore_index=True).drop_duplicates(subset="timestamp"))
    return full_df


In [74]:
usdc_luna_raw = get_coinbase_hourly_raw(
    "USDC-EUR",
    "2022-04-25",
    "2022-05-25"
)
usdc_luna_raw.head()

,timestamp,low,high,open,close,volume
0,1651744800,0.9420,0.9440,0.9434,0.9434,158735.34
1,1651741200,0.9430,0.9437,0.9433,0.9433,73838.72
2,1651737600,0.9434,0.9447,0.9435,0.9436,78373.82
3,1651734000,0.9420,0.9443,0.9434,0.9435,117519.87
4,1651730400,0.9417,0.9436,0.9425,0.9435,49331.70


In [75]:
usdc_luna_raw.shape

(721, 6)

In [76]:
usdc_luna_raw.to_csv(
    "../data/raw/hourly/coinbase/usdc_luna_hourly.csv",
    index=False
)

In [77]:
usdt_luna_raw = get_coinbase_hourly_raw(
    "USDT-EUR",
    "2022-04-25",
    "2022-05-25"
)
usdt_luna_raw.head()

,timestamp,low,high,open,close,volume
0,1651744800,0.9424,0.9441,0.9433,0.9430,339828.26
1,1651741200,0.9428,0.9438,0.9433,0.9431,122420.25
2,1651737600,0.9432,0.9446,0.9432,0.9434,133619.98
3,1651734000,0.9427,0.9443,0.9432,0.9432,181070.32
4,1651730400,0.9417,0.9433,0.9419,0.9432,176273.35


In [78]:
usdt_luna_raw.shape

(721, 6)

In [79]:
usdt_luna_raw.to_csv(
    "../data/raw/hourly/coinbase/usdt_luna_hourly.csv",
    index=False
)

In [80]:
usdt_celsius_raw = get_coinbase_hourly_raw(
    "USDT-EUR",
    "2022-05-29", 
    "2022-06-28"
)
usdt_celsius_raw.head()


,timestamp,low,high,open,close,volume
0,1654682400,0.93151,0.93379,0.93376,0.93251,243935.97
1,1654678800,0.93320,0.93640,0.93550,0.93379,168755.57
2,1654675200,0.93440,0.93596,0.93494,0.93560,124422.22
3,1654671600,0.93431,0.93560,0.93480,0.93491,128693.46
4,1654668000,0.93423,0.93501,0.93501,0.93479,25084.31


In [81]:
usdt_celsius_raw.shape

(721, 6)

In [82]:
usdt_celsius_raw.to_csv(
    "../data/raw/hourly/coinbase/usdt_celsius_hourly.csv",
    index=False
)

In [83]:
usdc_celsius_raw = get_coinbase_hourly_raw(
    "USDC-EUR",
    "2022-05-29", 
    "2022-06-28"
)
usdc_celsius_raw.head()

,timestamp,low,high,open,close,volume
0,1654682400,0.9318,0.9343,0.9342,0.9327,297236.16
1,1654678800,0.9335,0.9368,0.9365,0.9342,124340.78
2,1654675200,0.9350,0.9366,0.9356,0.9362,74225.83
3,1654671600,0.9349,0.9363,0.9356,0.9356,100101.62
4,1654668000,0.9347,0.9357,0.9357,0.9355,79247.69


In [84]:
usdc_celsius_raw.shape

(721, 6)

In [85]:
usdc_celsius_raw.to_csv(
    "../data/raw/hourly/coinbase/usdc_celsius_hourly.csv",
    index=False
)

In [86]:
usdt_svb_raw = get_coinbase_hourly_raw(
    "USDT-EUR",
    "2023-02-24",
    "2023-03-26" 
)
usdt_svb_raw.head()

,timestamp,low,high,open,close,volume
0,1678096800,0.93982,0.94105,0.94098,0.94070,576613.91
1,1678093200,0.93897,0.94086,0.93942,0.94086,138973.25
2,1678089600,0.93810,0.93950,0.93900,0.93950,156980.73
3,1678086000,0.93850,0.93933,0.93930,0.93894,66197.20
4,1678082400,0.93881,0.93976,0.93891,0.93930,72567.96


In [87]:
usdt_svb_raw.shape

(716, 6)

In [88]:
usdt_svb_raw.to_csv(
    "../data/raw/hourly/coinbase/usdt_svb_hourly.csv",
    index=False
)

In [89]:
usdc_svb_raw = get_coinbase_hourly_raw(
    "USDC-EUR",
    "2023-02-24",
    "2023-03-26" 
)
usdc_svb_raw.head()

,timestamp,low,high,open,close,volume
0,1678096800,0.9396,0.9410,0.9409,0.9405,38092.98
1,1678093200,0.9390,0.9406,0.9394,0.9406,69312.97
2,1678089600,0.9380,0.9394,0.9386,0.9394,22676.50
3,1678086000,0.9380,0.9396,0.9396,0.9389,6646.88
4,1678082400,0.9391,0.9398,0.9391,0.9394,1592.23


In [90]:
usdc_svb_raw.shape

(714, 6)

In [91]:
usdc_svb_raw.to_csv(
    "../data/raw/hourly/coinbase/usdc_svb_hourly.csv",
    index=False
)

In [92]:
usdc_election_raw = get_coinbase_hourly_raw(
    "USDC-EUR",
    "2024-10-23", 
    "2024-11-22"
)
usdc_election_raw.head()

,timestamp,low,high,open,close,volume
0,1730541600,0.9242,0.9244,0.9243,0.9242,150818.34
1,1730538000,0.9236,0.9244,0.9239,0.9242,1070697.25
2,1730534400,0.9237,0.9239,0.9238,0.9237,282715.02
3,1730530800,0.9237,0.9239,0.9237,0.9237,356241.85
4,1730527200,0.9236,0.9238,0.9237,0.9237,32619.50


In [93]:
usdc_election_raw.shape

(721, 6)

In [94]:
usdc_election_raw.to_csv(
    "../data/raw/hourly/coinbase/usdc_election_hourly.csv",
    index=False
)

In [95]:
usdt_election_raw = get_coinbase_hourly_raw(
    "USDT-EUR",
    "2024-10-23", 
    "2024-11-22"
)
usdt_election_raw.head()

,timestamp,low,high,open,close,volume
0,1730541600,0.92366,0.92403,0.92375,0.92403,78657.29
1,1730538000,0.92327,0.92380,0.92343,0.92369,60106.24
2,1730534400,0.92332,0.92360,0.92345,0.92342,40179.42
3,1730530800,0.92330,0.92368,0.92351,0.92345,267683.72
4,1730527200,0.92327,0.92363,0.92340,0.92361,133052.05


In [96]:
usdt_election_raw.shape

(721, 6)

In [97]:
usdt_election_raw.to_csv(
    "../data/raw/hourly/coinbase/usdt_election_hourly.csv",
    index=False
)

## 4. Hourly FX Benchmark — Dukascopy

USDT and USDC are designed to remain close to **$1**, but the Coinbase market data used in this project quotes both stablecoins in EUR.

Hourly EUR/USD data is therefore used as an independent foreign-exchange benchmark.

Later in the analysis, the two prices are combined as:

`stablecoin/EUR × EUR/USD = approximate stablecoin/USD`

This helps distinguish a movement in the stablecoin itself from a movement caused by changes in the EUR/USD exchange rate.

EUR/USD data is downloaded manually from Dukascopy Historical Data Export using 1-hour BID candles.

The downloaded files are stored unchanged in:

`data/raw/fx/`

Cleaning and timestamp standardization are performed in the next notebook.

## Data Ingestion Summary

The raw data layer now contains the inputs required for the project:

- **daily BTC, ETH, USDT and USDC data** for broader historical context;
- **hourly BTC and ETH data** for detailed market context around each event;
- **hourly USDT data** for the modelled operational balance;
- **hourly USDC data** as a comparison stablecoin;
- **hourly EUR/USD data** for interpreting stablecoin prices quoted in EUR.

No missing-value treatment, event interpretation or risk calculation is performed during ingestion.

The next step is to validate and standardize these datasets before using them in the event analyses.